# What the compiler does to a team in the `inline` build

The `inline` and `noipa` builds compile the same TPGs. The obvious difference is
inlining: in `inline` the body of each library function (`fixedpt_cos`,
`protected_fixedpt_div_32`, ...) is copied into every team that uses it, in `noipa`
the team calls it. This notebook shows that inlining is not the only difference.

**Claim.** Once the bodies are inlined, two of GCC's `-O1` passes reshape the team's
code across the boundary between library code and team code:

- **jump threading and the dominator optimisations** drop tests whose outcome is
  already known and duplicate a path specialised on it;
- **block reordering** moves rarely taken paths out of the team's label range.

In `noipa` the same passes run, but only inside each library function.

**How it is shown.** (1) One TPG read in both builds. (2) The same TPG rebuilt with
each pass turned off: what disappears with a pass is that pass's doing. (3) How
often it happens across the 161 TPGs. (4) What it costs the static model of
`model_comparison.ipynb`.

Two words used throughout: a team's **stored block** is the code between its two
labels, `T<n>_start` and `T<n>_end`, which is what the loader keeps and every other
notebook counts; its **reachable code** is everything it can execute between the two
labels, found by following its branches and jumps wherever the code was placed.


In [2]:
# --- Configuration -----------------------------------------------------------
TREE   = "/home/paulallaire/dev/modelization/tpg_nextflow"     # the pipeline's checkout
BUILDS = {"inline": "db/db_fixedpt_iset32_inline", "noipa": "db/db_fixedpt_iset32_noipa"}
UARCH  = "cv32e40x_im1_zba_zbb"

# The TPG of sections 1 and 2: the same seed directory exists in both builds.
SEED = ("useInstrTrig-True_useInstrLogExp-False_useInstrLog2Exp2-False_useInstrZmmul-False_"
        "useInstrExpensiveArithmetic-True_useInstrComparison-True_seed-13_instrType-fixedpt_iset32")
GENERATIONS = "100_gen"
THREADED, REORDERED = 6, 1          # one team shaped by each pass

# Section 2 rebuilds the inline TPG.c the way the pipeline compiles it, in its container.
CONTAINER = f"{TREE}/containers/x-heep.sif"
XHEEP     = f"{TREE}/x-heep"
APP       = f"{XHEEP}/sw/applications/tpg_modelization/tpg_inference_instrDispatch_instrTeams_instrTPG"
PASSES = {                           # name: flags added to the pipeline's -O1
    "as built (-O1)":       [],
    "no block reordering":  ["-fno-reorder-blocks"],
    "no jump threading":    ["-fno-thread-jumps"],
    "no dominator opts":    ["-fno-tree-dominator-opts"],
}
N_FOLDS, N_REPEATS, CV_SEED = 5, 3, 0     # the protocol of model_comparison.ipynb (section 4)
FALLBACK = {"call": "branch"}             # ...and its model for a class too small for its own

import sys, os, re, io, glob, shutil, tempfile, subprocess, contextlib
from collections import Counter
import numpy as np

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "classes")) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

from classes.database import Database
from analysis.disassembler import INSTRUCTION_LATENCY, _LINE_RE, _instruction_from_match
print("ready")


ready


In [3]:
# --- Reading a disassembly ---------------------------------------------------
LABEL  = re.compile(r"^([0-9a-f]{8}) <T(\d+)_(start|end)>:")
SYMBOL = re.compile(r"^([0-9a-f]{8}) <([^>]+)>:\s*$")
LINE   = re.compile(r"^\s*([0-9a-f]+):\s+[0-9a-f]+\s+(\S+)\s*([^<#]*?)\s*(?:<([^>]+)>)?\s*(?:#.*)?$")
TARGET = re.compile(r"(?:^|,)\s*([0-9a-f]+)\s*$")
BRANCH = {"beq", "bne", "blt", "bge", "bltu", "bgeu", "beqz", "bnez", "blez", "bgez",
          "bltz", "bgtz", "bgt", "ble", "bgtu", "bleu"}
CALLS  = {"jal", "jalr", "call", "tail"}

class Binary:
    '''The instructions of one objdump file, its team labels and its functions.'''
    def __init__(self, path):
        self.ins, self.labels, self.funcs, func = [], {}, {}, None
        for line in open(path):
            line = line.rstrip("\n")
            if (m := LABEL.match(line)):
                self.labels[(int(m.group(2)), m.group(3))] = int(m.group(1), 16)
                continue
            if (m := SYMBOL.match(line)):
                func = self.funcs.setdefault(m.group(2), [])
                continue
            if (m := LINE.match(line)):
                t = TARGET.search(m.group(3))
                op = _instruction_from_match(_LINE_RE.search(line)).operation
                self.ins.append((int(m.group(1), 16), m.group(2), m.group(3).strip(),
                                 int(t.group(1), 16) if t else None, op, m.group(4)))
                if func is not None:
                    func.append(len(self.ins) - 1)
        self.at = {a: i for i, (a, *_ ) in enumerate(self.ins)}
        self.teams = sorted({t for t, _ in self.labels})

    def team(self, t):
        '''(stored block, reachable code) of team t, as instruction indices.

        The stored block is what the loader keeps: the lines after T<t>_start up to
        the next end label. The reachable code follows every branch and jump from
        T<t>_start until T<t>_end, wherever the code was placed.'''
        s, e = self.labels[(t, "start")], self.labels[(t, "end")]
        e_block = min(a for (u, k), a in self.labels.items() if k == "end" and a > s)
        block = [i for i, (a, *_ ) in enumerate(self.ins) if s < a < e_block]
        seen, todo = set(), [self.at[s]]
        while todo:
            i = todo.pop()
            if i in seen or i >= len(self.ins):
                continue
            a, mn, _ops, tgt, _op, _sym = self.ins[i]
            if a == e:
                continue
            seen.add(i)
            if mn in ("jr", "ret", "jalr"):
                continue
            if mn in BRANCH and tgt in self.at:
                todo.append(self.at[tgt])
            if mn == "j":
                if tgt in self.at:
                    todo.append(self.at[tgt])
                continue
            todo.append(i + 1)
        return block, sorted(seen - {self.at[s]})          # the timing probe is not team code

    def show(self, idx, inside=None, only_control=False):
        '''Print instructions; mark those outside `inside` (the stored block).'''
        for i in idx:
            a, mn, ops, _t, _op, sym = self.ins[i]
            if mn in CALLS and sym:
                ops += f" <{sym}>"                         # name the callee of a call
            if only_control and mn not in BRANCH | {"j"} and not (inside is not None and i not in inside):
                continue
            mark = "   <- outside the team's labels" if inside is not None and i not in inside else ""
            print(f"    {a:6x}:  {mn:<7} {ops}{mark}")

    def cost(self, idx, _stack=()):
        '''The manual's cost of these instructions, a call counted with its callee's body.'''
        total = 0
        for i in idx:
            op = self.ins[i][4].lower()
            total += INSTRUCTION_LATENCY.get(op, INSTRUCTION_LATENCY.get(op.split("_")[0], 1))
            callee = op.partition("_")[2] if op.split("_")[0] in CALLS else ""
            if callee in self.funcs and callee not in _stack:
                total += self.cost(self.funcs[callee], _stack + (callee,))
        return total

def seed_dir(build):
    return next(p for p in sorted(glob.glob(f"{TREE}/armlearn-tpgs/*/{build}/**/{SEED}", recursive=True))
                if GENERATIONS in p)

def disassembly(tpg_dir):
    return sorted(glob.glob(f"{tpg_dir}/inference/results/{UARCH}_rv32*/"
                            "disassembly_tpg_inference_instrDispatch_instrTeams_instrTPG.txt"))[0]

DIS = {build: Binary(disassembly(seed_dir(build))) for build in BUILDS}
print({build: f"{len(b.teams)} teams" for build, b in DIS.items()})


{'inline': '12 teams', 'noipa': '12 teams'}


## 1. One team, two builds

Team 6 of this TPG runs five programs; two of them divide by the same input with
`protected_fixedpt_div_32`, which returns 0 when the divisor is too small
(|divisor| < 256) and divides otherwise. In `noipa` the team calls the function
twice, and each call tests that guard inside the function. In `inline` the two
bodies are copied into the team.


In [4]:
for build in ("noipa", "inline"):
    b = DIS[build]
    block, reach = b.team(THREADED)
    print(f"{build}: T{THREADED}, {len(block)} instructions between its labels, "
          f"{len(set(reach) - set(block))} more reachable outside them")
    b.show(reach, inside=set(block))
    print()
print("noipa: the function both calls go to")
DIS["noipa"].show(DIS["noipa"].funcs["protected_fixedpt_div_32"])


noipa: T6, 19 instructions between its labels, 0 more reachable outside them
      1408:  mv      s11,a5
      140c:  lw      a1,0(s5)
      1410:  lw      a0,4(s6)
      1414:  jal     560 <protected_fixedpt_div_32>
      1418:  lw      a5,4(s2)
      141c:  sub     a0,a0,a5
      1420:  sw      a0,96(sp)
      1424:  lw      a1,0(s5)
      1428:  lw      a0,0(s2)
      142c:  jal     560 <protected_fixedpt_div_32>
      1430:  sw      a0,100(sp)
      1434:  lw      a5,8(s6)
      1438:  sw      a5,104(sp)
      143c:  lw      a5,0(s2)
      1440:  max     a5,a5,zero
      1444:  sw      a5,108(sp)
      1448:  lw      a5,4(s2)
      144c:  neg     a5,a5
      1450:  sw      a5,112(sp)

inline: T6, 21 instructions between its labels, 6 more reachable outside them
      1ae8:  lw      t1,0(a2)
      1aec:  addi    t4,t1,255
      1af0:  li      t3,510
      1af4:  bgeu    t3,t4,2530
      1af8:  srai    t4,t1,0x8
      1afc:  lw      t3,4(a3)
      1b00:  lw      t1,4(a1)
      1b04: 

**Reading it.** In `inline`, three things are not inlining:

1. **One guard test for two divides.** The guard (`addi ...,255`, `li ...,510`,
   `bgeu`) appears once, then both `div` run. The second test is gone: on that path
   its outcome is already known.
2. **A duplicated, specialised path.** The block reached when the guard trips does
   not just skip a divide. It recomputes the rest of the program with the result
   fixed at 0: the `sub` that followed the divide becomes `neg`, and the second
   result becomes `li ...,0`.
3. **That path is placed outside the team's labels**, after another team, with a
   jump back. The stored block does not contain it.

In `noipa` none of this crosses the call: each call tests its own guard, and the
function's rare path (`li a0,0; ret`) is placed at the end of the function itself.


## 2. Which pass does it

The `inline` TPG is rebuilt in the pipeline's container, with the pipeline's
flags, once as it was built and once with each pass turned off. The rebuild as
built must give the measured binary back, or nothing below holds.


In [5]:
def rebuild(tpg_dir, passes):
    '''Compile the TPG's TPG.c as the pipeline does, once per set of extra flags.'''
    if not (shutil.which("apptainer") and os.path.exists(CONTAINER)):
        print("apptainer or the container is missing: the rebuilds are skipped")
        return {}
    work = tempfile.mkdtemp(prefix="passes_")
    for sub in ("codegen", "precalcul"):
        os.makedirs(f"{work}/{sub}")
    shutil.copytree(f"{APP}/fixedptc", f"{work}/fixedptc")
    shutil.copy(f"{APP}/approximateComputingTools.h", work)
    shutil.copy(f"{APP}/codegen/externHeader.h", f"{work}/codegen")
    for f in glob.glob(f"{tpg_dir}/outLogs/codegen_DispatchInstrumented_TeamsInstrumented/TPG*"):
        shutil.copy(f, f"{work}/codegen")
    shutil.copy(f"{tpg_dir}/outLogs/precalcul/LE_states.h", f"{work}/precalcul")
    gcc = ("/opt/tools/riscv/bin/riscv32-unknown-elf-gcc -march=rv32im_zicsr_zba_zbb -mabi=ilp32 "
           "-w -O1 -g -nostdlib -ffunction-sections -DHOST_BUILD -DON_CHIP -DINTERNAL_CRTO "
           f"-DUSE_FIXEDPT -I{work} -I{work}/codegen -I/xh/sw/device/lib/base")
    steps = [f"{gcc} {' '.join(flags)} -c {work}/codegen/TPG.c -o {work}/{k}.o && "
             f"/opt/tools/riscv/bin/riscv32-unknown-elf-objdump -d {work}/{k}.o > {work}/{k}.txt"
             for k, flags in enumerate(passes.values())]
    subprocess.run(["apptainer", "exec", "--bind", f"{XHEEP}:/xh", "--bind", f"{work}:{work}",
                    CONTAINER, "sh", "-c", " && ".join(steps)], check=True, capture_output=True)
    return {name: Binary(f"{work}/{k}.txt") for k, name in enumerate(passes)}

REB = rebuild(seed_dir("inline"), PASSES)
if REB:
    built, measured = REB["as built (-O1)"], DIS["inline"]
    same = [t for t in measured.teams
            if [built.ins[i][4] for i in built.team(t)[1]] == [measured.ins[i][4] for i in measured.team(t)[1]]]
    print(f"rebuilt as built: {len(same)} of {len(measured.teams)} teams identical to the measured binary\n")
    print(f"instructions outside the team's labels, per team")
    print(f"  {'team':>5}" + "".join(f"{name:>22}" for name in REB))
    for t in built.teams:
        print(f"  {t:>5}" + "".join(f"{len(set(r.team(t)[1]) - set(r.team(t)[0])):>22}" for r in REB.values()))
    guard = lambda r: sum(1 for i in r.team(THREADED)[1] if r.ins[i][1] == "li" and r.ins[i][2].endswith(",510"))
    print(f"\nguard tests in T{THREADED}: " + ", ".join(f"{name} {guard(r)}" for name, r in REB.items()))


rebuilt as built: 12 of 12 teams identical to the measured binary

instructions outside the team's labels, per team
   team        as built (-O1)   no block reordering     no jump threading     no dominator opts
      0                     0                     0                     0                     0
      1                    25                     0                    25                    25
      2                     0                     0                     0                     0
      3                    77                     0                    77                    77
      4                    27                     0                    27                    27
      5                    25                     0                    25                    25
      6                     6                     6                     0                     0
      7                     0                     0                     0                     0
      8             

In [5]:
if REB:
    for name in ("no jump threading", "no block reordering"):
        r = REB[name]
        block, reach = r.team(THREADED)
        print(f"T{THREADED}, {name}: {len(block)} instructions between its labels, "
              f"{len(set(reach) - set(block))} outside")
        r.show(reach, inside=set(block))
        print()


T6, no jump threading: 26 instructions between its labels, 0 outside
       fe8:  lw      t5,0(a2)
       fec:  addi    t6,t5,255
       ff0:  li      t3,510
       ff4:  li      t1,0
       ff8:  bgeu    t3,t6,100c
       ffc:  lw      t1,4(a1)
      1000:  slli    t1,t1,0x8
      1004:  srai    t3,t5,0x8
      1008:  div     t1,t1,t3
      100c:  lw      t3,4(a3)
      1010:  sub     t1,t1,t3
      1014:  sw      t1,96(sp)
      1018:  lw      t4,0(a3)
      101c:  li      t2,510
      1020:  li      t1,0
      1024:  bgeu    t2,t6,1034
      1028:  slli    t1,t4,0x8
      102c:  srai    t5,t5,0x8
      1030:  div     t1,t1,t5
      1034:  sw      t1,100(sp)
      1038:  lw      t1,8(a1)
      103c:  sw      t1,104(sp)
      1040:  max     t4,t4,zero
      1044:  sw      t4,108(sp)
      1048:  neg     t3,t3
      104c:  sw      t3,112(sp)

T6, no block reordering: 21 instructions between its labels, 6 outside
       fb8:  lw      t1,0(a2)
       fbc:  addi    t4,t1,255
       fc0:  

**Reading it.** Each team's code outside its labels has one cause:

- **T6 is shaped by jump threading and the dominator optimisations.** Turn either
  off and the team goes back to the source's structure: two guard tests, one per
  divide, no duplicated path, nothing outside the labels. Turning block reordering
  off changes nothing for it: the duplicated path is still placed away.
- **The other teams with code outside their labels are shaped by block reordering.**
  Turning it off brings all of it back between the labels; the threading flags
  leave them as they are. Team 1 below carries an inlined `fixedpt_cos`, whose range
  reduction tests the sign and the quadrant of its argument.


In [6]:
if REB:
    for name in ("as built (-O1)", "no block reordering"):
        r = REB[name]
        block, reach = r.team(REORDERED)
        print(f"T{REORDERED}, {name}: {len(block)} instructions between its labels, "
              f"{len(set(reach) - set(block))} outside (branches, jumps and the code outside shown)")
        r.show(reach, inside=set(block), only_control=True)
        print()


T1, as built (-O1): 79 instructions between its labels, 25 outside (branches, jumps and the code outside shown)
       150:  bltz    t1,2c8
       168:  bltu    t4,t3,2d0
       2c8:  add     t1,t1,t4   <- outside the team's labels
       2cc:  j       154   <- outside the team's labels
       2d0:  lui     t3,0xfffce   <- outside the team's labels
       2d4:  addi    t3,t3,-1088   <- outside the team's labels
       2d8:  add     t3,t1,t3   <- outside the team's labels
       2dc:  lui     t4,0x19   <- outside the team's labels
       2e0:  addi    t4,t4,543   <- outside the team's labels
       2e4:  bltu    t4,t3,300   <- outside the team's labels
       2e8:  lui     t3,0xfffce   <- outside the team's labels
       2ec:  addi    t3,t3,-1087   <- outside the team's labels
       2f0:  add     t1,t1,t3   <- outside the team's labels
       2f4:  li      t3,-1   <- outside the team's labels
       2f8:  mv      s0,t3   <- outside the team's labels
       2fc:  j       180   <- outsid

**Reading it.** As built, the tests of the angle's sign and quadrant branch to code
placed after the team's end label, which jumps back; without block reordering the
same instructions stay between the labels (one jump fewer: `bltz` to the moved
`add` becomes `bgez` over it). Same computation, different placement.


## 3. How often, across the 161 TPGs

Every team of every binary, in both builds: how many reach code outside their labels,
and how many teams the loader loses because two teams' labels overlap (a team whose
end label comes after the next team's start label has no stored block at all).


In [7]:
for build in BUILDS:
    files = sorted(glob.glob(f"{TREE}/armlearn-tpgs/*/{build}/**/inference/results/{UARCH}_rv32*/"
                             "disassembly_tpg_inference_instrDispatch_instrTeams_instrTPG.txt", recursive=True))
    teams = moved = outside = reached = lost = 0
    for f in files:
        b = Binary(f)
        starts = sorted((b.labels[(t, "start")], t) for t in b.teams)
        for (s, t), nxt in zip(starts, starts[1:] + [(None, None)]):
            if nxt[0] is not None and nxt[0] < b.labels[(t, "end")]:
                lost += 1                       # the next team opens before this one closes
                continue
            block, reach = b.team(t)
            out = len(set(reach) - set(block))
            teams += 1; moved += out > 0; outside += out; reached += len(reach)
    print(f"{build:<7} {len(files)} TPGs, {teams} teams read: {moved} reach code outside their labels "
          f"({outside} instructions, {100 * outside / reached:.1f}% of all the code they reach); "
          f"{lost} teams lost to overlapping labels")


inline  161 TPGs, 3284 teams read: 758 reach code outside their labels (18547 instructions, 8.4% of all the code they reach); 10 teams lost to overlapping labels
noipa   161 TPGs, 3294 teams read: 3 reach code outside their labels (6 instructions, 0.0% of all the code they reach); 0 teams lost to overlapping labels


## 4. What it costs the static model

The mechanistic OLS of `model_comparison.ipynb` (the manual's estimate, a call
counted with its callee's body, rescaled by a fitted slope and intercept), with the
same classes, folds and scores; with one model per class, the few `inline` teams that
still call a function are predicted by the branch model, as there. For `inline` it
is computed twice: on the stored block, as everywhere else, and on the reachable
code. `noipa` is the reference.
Classes are each build's own, so only "straight-line" and "all teams" hold the same
teams in both builds.


In [8]:
CLASSES = ("straight", "branch", "call")
LABEL_OF = {"straight": "straight-line", "branch": "branches, no call", "call": "calls a function"}

def estimates(build):
    '''Measured latency, class, TPG and the two estimates of every measured team.'''
    with contextlib.redirect_stdout(io.StringIO()):
        db = Database.load(BUILDS[build])
    rows, cache = [], {}
    for tpg, tid, ct, m in db.get_measurements_for_uarch(UARCH):
        if not ct.instructions:
            continue
        if tpg.source_path not in cache:
            cache[tpg.source_path] = Binary(disassembly(tpg.source_path))
        b = cache[tpg.source_path]
        block, reach = b.team(tid)
        mn = {i.operation.lower().split("_")[0] for i in ct.instructions}
        rows.append((m.latency, "call" if mn & CALLS else "branch" if mn & (BRANCH | {"j", "jr", "ret"})
                     else "straight", tpg.source_path, b.cost(block), b.cost(reach)))
    y, cls, tpg, block, reach = (np.array(v) for v in zip(*rows))
    return dict(y=y.astype(float), cls=cls, tpg=tpg, block=block.astype(float), reach=reach.astype(float))

def scores(d, est, per_class):
    '''Out-of-fold MAPE and R2 of a x est + b, folds keeping every TPG whole.'''
    out = {c: [] for c in CLASSES + ("all",)}
    for rep in range(N_REPEATS):
        rng = np.random.default_rng(CV_SEED + rep)
        tpgs = np.unique(d["tpg"]); rng.shuffle(tpgs)
        fold = dict(zip(tpgs, np.arange(len(tpgs)) % N_FOLDS))
        f = np.array([fold[t] for t in d["tpg"]])
        p = np.full(len(d["y"]), np.nan)
        for k in range(N_FOLDS):
            for c in (CLASSES if per_class else (None,)):
                tr = (f != k) & ((d["cls"] == c) if c else True)
                te = (f == k) & ((d["cls"] == c) if c else True)
                if tr.sum() < 30 and c in FALLBACK:          # too few teams: borrow a class's model
                    tr = (f != k) & (d["cls"] == FALLBACK[c])
                if tr.sum() < 30 or not te.any():
                    continue
                beta = np.polyfit(est[tr], d["y"][tr], 1)
                p[te] = np.polyval(beta, est[te])
        for c in out:
            m = (d["cls"] == c) if c != "all" else np.ones(len(d["y"]), bool)
            if m.sum() >= 30 and not np.isnan(p[m]).any():
                y = d["y"][m]
                out[c].append((np.mean(np.abs(p[m] - y) / y) * 100,
                               1 - ((y - p[m]) ** 2).sum() / ((y - y.mean()) ** 2).sum()))
    return {c: tuple(np.mean(v, axis=0)) if v else None for c, v in out.items()}

D = {build: estimates(build) for build in BUILDS}
cases = (("inline, stored block", D["inline"], D["inline"]["block"]),
         ("inline, reachable code", D["inline"], D["inline"]["reach"]),
         ("noipa, stored block", D["noipa"], D["noipa"]["block"]))
for title, per_class in (("one model per class", True), ("one model for all teams", False)):
    print(f"mechanistic OLS, {title}: MAPE / R2")
    print(f"  {'':<24}" + "".join(f"{LABEL_OF.get(c, 'all teams'):>22}" for c in CLASSES + ("all",)))
    for name, d, est in cases:
        r = scores(d, est, per_class)
        print(f"  {name:<24}" + "".join(f"{f'{r[c][0]:.1f}% / {r[c][1]:.3f}' if r[c] else '-':>22}"
                                        for c in CLASSES + ("all",)))
    print()


mechanistic OLS, one model per class: MAPE / R2
                                   straight-line     branches, no call      calls a function             all teams
  inline, stored block              9.6% / 0.977         76.4% / 0.543                     -         35.4% / 0.665
  inline, reachable code            9.6% / 0.977         74.4% / 0.588                     -         34.6% / 0.697
  noipa, stored block               8.9% / 0.980          7.7% / 0.962         79.8% / 0.515         30.1% / 0.710

mechanistic OLS, one model for all teams: MAPE / R2
                                   straight-line     branches, no call      calls a function             all teams
  inline, stored block             30.6% / 0.731         76.4% / 0.545                     -         48.4% / 0.660
  inline, reachable code           30.7% / 0.691         75.2% / 0.589                     -         47.9% / 0.692
  noipa, stored block              30.1% / 0.698         22.4% / 0.663         79.8% / 0.517  

**Reading it.** The stored-block rows are the mechanistic OLS of
`model_comparison.ipynb`. Reading `inline` teams by their reachable code adds back
what the passes placed outside the labels. With one model per class only the branch
teams change, since they carry the inlined library code; with one model for all
teams the straight-line teams move too, because they share its slope. What remains between `inline` and `noipa` is not the
passes: in both builds a static count includes every path of a library function,
while only one runs and which one depends on the argument (`mechanistic_model.ipynb`,
section 6).

## Conclusion

- **The two builds differ by more than inlining.** In `inline`, jump threading and
  the dominator optimisations merge tests and duplicate paths across the inlined
  library code, and block reordering moves its rare paths out of the team's labels.
  Each effect disappears when its pass is turned off (section 2).
- **`noipa` confines these passes to each library function.** The team's code stays
  between its labels, and each function exists once, in one piece.
- **The cost of the passes to a static model is measured** (section 4): the code
  they move out of the labels. The larger gap between the builds, and the ceiling
  of every model, comes from the argument-dependent cost of the library functions.
